In [1]:
from pathlib import Path
from datetime import datetime
import getpass
import json
import re
import warnings

import numpy as np
import pandas as pd
import psycopg2
from psycopg2 import sql

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)
from sklearn.base import clone

import joblib

warnings.filterwarnings("ignore")

PROJECT_PATH = Path(
    r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM"
)

PREDICTIONS_PATH = PROJECT_PATH / "data" / "predictions"
MODELS_PATH = PROJECT_PATH / "models"

PREDICTIONS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

DB_NAME = "ecommerce_ai_db"
DB_HOST = "localhost"
DB_PORT = 5432

print("Notebook 11: Customer Churn Prediction")
print(f"Project path: {PROJECT_PATH}")
print(f"Database: {DB_NAME}")
print(f"Prediction output path: {PREDICTIONS_PATH}")
print(f"Model output path: {MODELS_PATH}")

Notebook 11: Customer Churn Prediction
Project path: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM
Database: ecommerce_ai_db
Prediction output path: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions
Model output path: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\models


In [2]:
DB_USER = input("Enter PostgreSQL username: ").strip()
DB_PASSWORD = getpass.getpass("Enter PostgreSQL password: ")

if not DB_USER:
    raise ValueError(
        "PostgreSQL username cannot be empty."
    )

if not DB_PASSWORD:
    raise ValueError(
        "PostgreSQL password cannot be empty."
    )

connection = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)

connection.autocommit = False

print("PostgreSQL connection successful.")

Enter PostgreSQL username:  postgres
Enter PostgreSQL password:  ········


PostgreSQL connection successful.


In [3]:
def quote_identifier(identifier):
    """
    Safely quote a PostgreSQL identifier.
    """
    return sql.Identifier(identifier)


def read_sql(query, params=None):
    """
    Execute a SQL query and return a pandas DataFrame.
    """
    return pd.read_sql_query(
        query,
        connection,
        params=params
    )


def get_table_columns(schema_name, table_name):
    """
    Return metadata for all columns in a PostgreSQL table.
    """
    query = """
        SELECT
            table_schema,
            table_name,
            ordinal_position,
            column_name,
            data_type,
            udt_name,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name = %s
        ORDER BY ordinal_position
    """

    return read_sql(
        query,
        params=[schema_name, table_name]
    )


def normalize_text(value):
    """
    Normalize text for metadata comparison.
    """
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).lower()
    ).strip("_")


def is_numeric_type(data_type):
    return data_type in {
        "smallint",
        "integer",
        "bigint",
        "numeric",
        "decimal",
        "real",
        "double precision"
    }


def is_datetime_type(data_type):
    return data_type in {
        "date",
        "timestamp without time zone",
        "timestamp with time zone"
    }


def is_textual_type(data_type):
    return data_type in {
        "character varying",
        "character",
        "text",
        "uuid"
    }


print("Database helper functions created.")

Database helper functions created.


In [4]:
tables_df = read_sql(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN (
          'pg_catalog',
          'information_schema'
      )
    ORDER BY
        table_schema,
        table_name
    """
)

if tables_df.empty:
    raise RuntimeError(
        "No user-created PostgreSQL tables were discovered."
    )

print(
    f"Discovered {len(tables_df)} PostgreSQL tables."
)

display(tables_df)

Discovered 30 PostgreSQL tables.


,table_schema,table_name,table_type
0,analytics,clv_overall_summary,BASE TABLE
1,analytics,clv_segment_summary,BASE TABLE
2,analytics,clv_validation,BASE TABLE
3,analytics,customer_lifetime_value,BASE TABLE
4,analytics,customer_segmentation,BASE TABLE
5,analytics,customer_segmentation_model_evaluation,BASE TABLE
6,analytics,segment_distribution,BASE TABLE
7,analytics,segment_feature_means,BASE TABLE
8,analytics,segment_summary,BASE TABLE
9,feature_engineered,category_features,BASE TABLE


In [5]:
all_columns = []

for _, table_row in tables_df.iterrows():
    schema_name = table_row["table_schema"]
    table_name = table_row["table_name"]

    table_columns = get_table_columns(
        schema_name,
        table_name
    )

    if not table_columns.empty:
        all_columns.append(table_columns)

if not all_columns:
    raise RuntimeError(
        "No PostgreSQL column metadata could be discovered."
    )

columns_df = pd.concat(
    all_columns,
    ignore_index=True
)

columns_df["normalized_column_name"] = (
    columns_df["column_name"]
    .map(normalize_text)
)

print(
    f"Discovered {len(columns_df)} columns "
    "across all user-created tables."
)

display(columns_df)

Discovered 314 columns across all user-created tables.


,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable,normalized_column_name
0,analytics,clv_overall_summary,1,metric,text,text,YES,metric
1,analytics,clv_overall_summary,2,value,text,text,YES,value
2,analytics,clv_segment_summary,1,clv_segment,text,text,YES,clv_segment
3,analytics,clv_segment_summary,2,customer_count,bigint,int8,YES,customer_count
4,analytics,clv_segment_summary,3,total_historical_clv,double precision,float8,YES,total_historical_clv
...,...,...,...,...,...,...,...,...
309,sales_analytics,overall_sales_kpis,9,total_order_value,numeric,numeric,YES,total_order_value
310,sales_analytics,overall_sales_kpis,10,average_order_value,numeric,numeric,YES,average_order_value
311,sales_analytics,overall_sales_kpis,11,average_delivery_days,numeric,numeric,YES,average_delivery_days
312,sales_analytics,overall_sales_kpis,12,late_delivery_rate,numeric,numeric,YES,late_delivery_rate


In [6]:
CUSTOMER_TOKENS = {
    "customer",
    "client",
    "buyer",
    "user",
    "shopper",
    "consumer",
    "member",
    "account"
}

DATE_TOKENS = {
    "date",
    "time",
    "timestamp",
    "created",
    "purchase",
    "order",
    "transaction",
    "placed",
    "completed"
}

def token_score(column_name, tokens):
    """
    Score a column based on semantic metadata tokens.
    This is discovery only, not a hardcoded column name.
    """
    normalized = normalize_text(column_name)

    pieces = set(normalized.split("_"))

    return len(pieces.intersection(tokens))


def discover_candidate_tables(columns_metadata):
    """
    Discover tables that could support customer churn modeling.
    """
    candidates = []

    for (schema_name, table_name), group in (
        columns_metadata.groupby(
            ["table_schema", "table_name"]
        )
    ):
        candidate_customer_columns = []

        candidate_date_columns = []

        for _, row in group.iterrows():
            column_name = row["column_name"]
            data_type = row["data_type"]

            customer_score = token_score(
                column_name,
                CUSTOMER_TOKENS
            )

            date_score = token_score(
                column_name,
                DATE_TOKENS
            )

            if (
                customer_score > 0
                and (
                    is_textual_type(data_type)
                    or is_numeric_type(data_type)
                )
            ):
                candidate_customer_columns.append(
                    {
                        "column_name": column_name,
                        "score": customer_score
                    }
                )

            if (
                date_score > 0
                and is_datetime_type(data_type)
            ):
                candidate_date_columns.append(
                    {
                        "column_name": column_name,
                        "score": date_score
                    }
                )

        if (
            candidate_customer_columns
            and candidate_date_columns
        ):
            candidates.append(
                {
                    "table_schema": schema_name,
                    "table_name": table_name,
                    "customer_candidates": candidate_customer_columns,
                    "date_candidates": candidate_date_columns
                }
            )

    return candidates


candidate_tables = discover_candidate_tables(
    columns_df
)

if not candidate_tables:
    raise RuntimeError(
        "No table was discovered containing both "
        "a possible customer identifier and a "
        "possible datetime column."
    )

print(
    f"Discovered {len(candidate_tables)} "
    "potential churn source tables."
)

for candidate in candidate_tables:
    print(
        f"\n{candidate['table_schema']}."
        f"{candidate['table_name']}"
    )
    print(
        "Customer candidates:",
        candidate["customer_candidates"]
    )
    print(
        "Date candidates:",
        candidate["date_candidates"]
    )

Discovered 3 potential churn source tables.

feature_engineered.customer_features
Customer candidates: [{'column_name': 'customer_id', 'score': 1}, {'column_name': 'customer_unique_id', 'score': 1}, {'column_name': 'customer_zip_code_prefix', 'score': 1}, {'column_name': 'customer_city', 'score': 1}, {'column_name': 'customer_state', 'score': 1}, {'column_name': 'customer_lifetime_days', 'score': 1}]
Date candidates: [{'column_name': 'first_purchase_date', 'score': 2}, {'column_name': 'last_purchase_date', 'score': 2}]

feature_engineered.order_features
Customer candidates: [{'column_name': 'customer_id', 'score': 1}]
Date candidates: [{'column_name': 'order_purchase_timestamp', 'score': 3}, {'column_name': 'order_approved_at', 'score': 1}, {'column_name': 'order_delivered_carrier_date', 'score': 2}, {'column_name': 'order_delivered_customer_date', 'score': 2}, {'column_name': 'order_estimated_delivery_date', 'score': 2}, {'column_name': 'purchase_date', 'score': 2}, {'column_name': 'p

In [7]:
def get_table_row_count(schema_name, table_name):
    query = sql.SQL(
        "SELECT COUNT(*) AS row_count FROM {}.{}"
    ).format(
        quote_identifier(schema_name),
        quote_identifier(table_name)
    )

    return int(
        pd.read_sql_query(
            query.as_string(connection),
            connection
        ).iloc[0]["row_count"]
    )


def profile_column(
    schema_name,
    table_name,
    column_name
):
    query = sql.SQL(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT({column}) AS non_null_rows,
            COUNT(DISTINCT {column}) AS distinct_values,
            MIN({column}) AS minimum_value,
            MAX({column}) AS maximum_value
        FROM {schema}.{table}
        """
    ).format(
        column=quote_identifier(column_name),
        schema=quote_identifier(schema_name),
        table=quote_identifier(table_name)
    )

    return pd.read_sql_query(
        query.as_string(connection),
        connection
    ).iloc[0]


validated_candidates = []

for candidate in candidate_tables:

    schema_name = candidate["table_schema"]
    table_name = candidate["table_name"]

    row_count = get_table_row_count(
        schema_name,
        table_name
    )

    if row_count == 0:
        continue

    for customer_candidate in candidate[
        "customer_candidates"
    ]:

        customer_column = customer_candidate[
            "column_name"
        ]

        customer_profile = profile_column(
            schema_name,
            table_name,
            customer_column
        )

        if (
            customer_profile["distinct_values"] < 2
        ):
            continue

        for date_candidate in candidate[
            "date_candidates"
        ]:

            date_column = date_candidate[
                "column_name"
            ]

            date_profile = profile_column(
                schema_name,
                table_name,
                date_column
            )

            if (
                pd.isna(date_profile["minimum_value"])
                or pd.isna(date_profile["maximum_value"])
            ):
                continue

            validated_candidates.append(
                {
                    "table_schema": schema_name,
                    "table_name": table_name,
                    "customer_column": customer_column,
                    "date_column": date_column,
                    "row_count": row_count,
                    "customer_distinct_values": int(
                        customer_profile[
                            "distinct_values"
                        ]
                    ),
                    "date_min": date_profile[
                        "minimum_value"
                    ],
                    "date_max": date_profile[
                        "maximum_value"
                    ]
                }
            )

validated_candidates_df = pd.DataFrame(
    validated_candidates
)

if validated_candidates_df.empty:
    raise RuntimeError(
        "Candidate tables were discovered, but none "
        "passed basic data validation."
    )

display(validated_candidates_df)

,table_schema,table_name,customer_column,date_column,row_count,customer_distinct_values,date_min,date_max
0,feature_engineered,customer_features,customer_id,first_purchase_date,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18
1,feature_engineered,customer_features,customer_id,last_purchase_date,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18
2,feature_engineered,customer_features,customer_unique_id,first_purchase_date,99441,96096,2016-09-04 21:15:19,2018-10-17 17:30:18
3,feature_engineered,customer_features,customer_unique_id,last_purchase_date,99441,96096,2016-09-04 21:15:19,2018-10-17 17:30:18
4,feature_engineered,customer_features,customer_zip_code_prefix,first_purchase_date,99441,14994,2016-09-04 21:15:19,2018-10-17 17:30:18
5,feature_engineered,customer_features,customer_zip_code_prefix,last_purchase_date,99441,14994,2016-09-04 21:15:19,2018-10-17 17:30:18
6,feature_engineered,customer_features,customer_city,first_purchase_date,99441,4119,2016-09-04 21:15:19,2018-10-17 17:30:18
7,feature_engineered,customer_features,customer_city,last_purchase_date,99441,4119,2016-09-04 21:15:19,2018-10-17 17:30:18
8,feature_engineered,customer_features,customer_state,first_purchase_date,99441,27,2016-09-04 21:15:19,2018-10-17 17:30:18
9,feature_engineered,customer_features,customer_state,last_purchase_date,99441,27,2016-09-04 21:15:19,2018-10-17 17:30:18


In [8]:
def calculate_candidate_score(row):
    """
    Rank validated candidates using observed data quality.
    """
    score = 0

    if row["row_count"] >= 100:
        score += 2

    if row["customer_distinct_values"] >= 50:
        score += 2

    if row["customer_distinct_values"] >= 1000:
        score += 1

    if (
        row["date_max"] > row["date_min"]
    ):
        score += 2

    return score


validated_candidates_df[
    "candidate_score"
] = validated_candidates_df.apply(
    calculate_candidate_score,
    axis=1
)

validated_candidates_df = (
    validated_candidates_df
    .sort_values(
        [
            "candidate_score",
            "customer_distinct_values",
            "row_count"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(validated_candidates_df)

,table_schema,table_name,customer_column,date_column,row_count,customer_distinct_values,date_min,date_max,candidate_score
0,feature_engineered,customer_features,customer_id,first_purchase_date,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18,7
1,feature_engineered,customer_features,customer_id,last_purchase_date,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18,7
2,feature_engineered,order_features,customer_id,order_purchase_timestamp,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18,7
3,feature_engineered,order_features,customer_id,order_approved_at,99441,99441,2016-09-15 12:16:38,2018-10-17 17:30:18,7
4,feature_engineered,order_features,customer_id,order_delivered_carrier_date,99441,99441,2016-10-04 10:26:40,2018-10-17 17:30:18,7
5,feature_engineered,order_features,customer_id,order_delivered_customer_date,99441,99441,2016-09-30 00:00:00,2018-11-12 00:00:00,7
6,feature_engineered,order_features,customer_id,order_estimated_delivery_date,99441,99441,2016-09-30 00:00:00,2018-11-12 00:00:00,7
7,feature_engineered,order_features,customer_id,purchase_date,99441,99441,2016-09-04,2018-10-17,7
8,feature_engineered,order_features,customer_id,purchase_month_start,99441,99441,2016-09-01 00:00:00,2018-10-01 00:00:00,7
9,public,customer_churn_predictions,customer_identifier,first_order_date,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18,7


In [12]:
# Cell 9 — Deep validation and safe source-table selection

def inspect_candidate_table_structure(
    candidate_row
):
    schema_name = candidate_row["table_schema"]
    table_name = candidate_row["table_name"]
    customer_column = candidate_row["customer_column"]
    date_column = candidate_row["date_column"]

    table_columns = get_table_columns(
        schema_name,
        table_name
    )

    if table_columns.empty:
        raise RuntimeError(
            f"No column metadata found for "
            f"{schema_name}.{table_name}."
        )

    # Create the normalized column names here.
    # get_table_columns() returns raw metadata and
    # does not automatically create this column.
    table_columns[
        "normalized_column_name"
    ] = (
        table_columns[
            "column_name"
        ]
        .map(normalize_text)
    )

    normalized_columns = (
        table_columns[
            "normalized_column_name"
        ]
        .tolist()
    )

    # Identify possible transaction/order/event columns
    transaction_tokens = {
        "order",
        "transaction",
        "purchase",
        "item",
        "event",
        "activity",
        "sale",
        "payment"
    }

    transaction_column_score = 0

    for normalized_column in normalized_columns:

        column_tokens = set(
            normalized_column.split("_")
        )

        transaction_column_score += len(
            column_tokens.intersection(
                transaction_tokens
            )
        )

    # Identify actual numeric columns
    numeric_columns = table_columns[
        table_columns[
            "data_type"
        ].apply(is_numeric_type)
    ]

    numeric_column_count = len(
        numeric_columns
    )

    # Query actual table statistics
    query = sql.SQL(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {customer}) AS distinct_customers,
            COUNT(DISTINCT DATE({date})) AS distinct_dates,
            MIN({date}) AS min_date,
            MAX({date}) AS max_date
        FROM {schema}.{table}
        WHERE {customer} IS NOT NULL
          AND {date} IS NOT NULL
        """
    ).format(
        customer=quote_identifier(
            customer_column
        ),
        date=quote_identifier(
            date_column
        ),
        schema=quote_identifier(
            schema_name
        ),
        table=quote_identifier(
            table_name
        )
    )

    statistics = pd.read_sql_query(
        query.as_string(connection),
        connection
    ).iloc[0]

    total_rows = int(
        statistics[
            "total_rows"
        ]
    )

    distinct_customers = int(
        statistics[
            "distinct_customers"
        ]
    )

    distinct_dates = int(
        statistics[
            "distinct_dates"
        ]
    )

    min_date = pd.to_datetime(
        statistics[
            "min_date"
        ]
    )

    max_date = pd.to_datetime(
        statistics[
            "max_date"
        ]
    )

    date_span_days = (
        max_date - min_date
    ).days

    # Validate historical coverage
    has_temporal_history = (
        date_span_days >= 180
        and distinct_dates >= 30
    )

    # Validate customer population
    has_customer_population = (
        distinct_customers >= 50
    )

    # Create a source-quality score
    source_quality_score = 0

    if has_temporal_history:
        source_quality_score += 10

    if has_customer_population:
        source_quality_score += 10

    if total_rows > distinct_customers:
        source_quality_score += 5

    if transaction_column_score > 0:
        source_quality_score += 5

    if numeric_column_count > 0:
        source_quality_score += 2

    return {
        "table_schema": schema_name,
        "table_name": table_name,
        "customer_column": customer_column,
        "date_column": date_column,
        "total_rows": total_rows,
        "distinct_customers": distinct_customers,
        "distinct_dates": distinct_dates,
        "min_date": min_date,
        "max_date": max_date,
        "date_span_days": date_span_days,
        "transaction_column_score": (
            transaction_column_score
        ),
        "numeric_column_count": (
            numeric_column_count
        ),
        "has_temporal_history": (
            has_temporal_history
        ),
        "has_customer_population": (
            has_customer_population
        ),
        "source_quality_score": (
            source_quality_score
        )
    }


candidate_structure_results = []

for _, candidate_row in (
    validated_candidates_df.iterrows()
):

    result = (
        inspect_candidate_table_structure(
            candidate_row
        )
    )

    candidate_structure_results.append(
        result
    )


candidate_structure_df = pd.DataFrame(
    candidate_structure_results
)

candidate_structure_df = (
    candidate_structure_df
    .sort_values(
        [
            "source_quality_score",
            "transaction_column_score",
            "numeric_column_count",
            "total_rows",
            "distinct_customers",
            "date_span_days"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Deep source-table validation results:"
)

display(
    candidate_structure_df
)


# Keep only candidates suitable for
# historical churn modeling
suitable_candidates = (
    candidate_structure_df[
        (
            candidate_structure_df[
                "has_temporal_history"
            ]
        )
        &
        (
            candidate_structure_df[
                "has_customer_population"
            ]
        )
    ]
    .copy()
)


if suitable_candidates.empty:

    raise RuntimeError(
        "No candidate table contains both "
        "a sufficiently large customer population "
        "and enough historical time coverage "
        "for churn modeling."
    )


# The first row is the strongest validated
# candidate according to actual observed data.
top_candidate = (
    suitable_candidates.iloc[0]
)


SOURCE_SCHEMA = (
    top_candidate[
        "table_schema"
    ]
)

SOURCE_TABLE = (
    top_candidate[
        "table_name"
    ]
)

CUSTOMER_COLUMN = (
    top_candidate[
        "customer_column"
    ]
)

DATE_COLUMN = (
    top_candidate[
        "date_column"
    ]
)


print(
    "\nValidated churn source selected:"
)

print(
    f"Schema: {SOURCE_SCHEMA}"
)

print(
    f"Table: {SOURCE_TABLE}"
)

print(
    f"Customer identifier: {CUSTOMER_COLUMN}"
)

print(
    f"Date column: {DATE_COLUMN}"
)

print(
    "Rows:",
    f"{top_candidate['total_rows']:,}"
)

print(
    "Distinct customers:",
    f"{top_candidate['distinct_customers']:,}"
)

print(
    "Distinct dates:",
    f"{top_candidate['distinct_dates']:,}"
)

print(
    "Date range:",
    top_candidate["min_date"],
    "to",
    top_candidate["max_date"]
)

print(
    "Source quality score:",
    top_candidate[
        "source_quality_score"
    ]
)

Deep source-table validation results:


,table_schema,table_name,customer_column,date_column,total_rows,distinct_customers,distinct_dates,min_date,max_date,date_span_days,transaction_column_score,numeric_column_count,has_temporal_history,has_customer_population,source_quality_score
0,feature_engineered,customer_features,customer_unique_id,first_purchase_date,99441,96096,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
1,feature_engineered,customer_features,customer_unique_id,last_purchase_date,99441,96096,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
2,feature_engineered,customer_features,customer_zip_code_prefix,first_purchase_date,99441,14994,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
3,feature_engineered,customer_features,customer_zip_code_prefix,last_purchase_date,99441,14994,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
4,feature_engineered,customer_features,customer_city,first_purchase_date,99441,4119,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
5,feature_engineered,customer_features,customer_city,last_purchase_date,99441,4119,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,4,15,True,True,32
6,feature_engineered,order_features,customer_id,order_delivered_customer_date,99441,99441,668,2016-09-30 00:00:00,2018-11-12 00:00:00,773,23,23,True,True,27
7,feature_engineered,order_features,customer_id,order_estimated_delivery_date,99441,99441,459,2016-09-30 00:00:00,2018-11-12 00:00:00,773,23,23,True,True,27
8,feature_engineered,order_features,customer_id,purchase_date,99441,99441,634,2016-09-04 00:00:00,2018-10-17 00:00:00,773,23,23,True,True,27
9,feature_engineered,order_features,customer_id,order_purchase_timestamp,99441,99441,634,2016-09-04 21:15:19,2018-10-17 17:30:18,772,23,23,True,True,27



Validated churn source selected:
Schema: feature_engineered
Table: customer_features
Customer identifier: customer_unique_id
Date column: first_purchase_date
Rows: 99,441
Distinct customers: 96,096
Distinct dates: 634
Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
Source quality score: 32


In [13]:
source_query = sql.SQL(
    """
    SELECT *
    FROM {schema}.{table}
    """
).format(
    schema=quote_identifier(SOURCE_SCHEMA),
    table=quote_identifier(SOURCE_TABLE)
)

source_df = pd.read_sql_query(
    source_query.as_string(connection),
    connection
)

if source_df.empty:
    raise RuntimeError(
        "The validated source table returned zero rows."
    )

source_df[DATE_COLUMN] = pd.to_datetime(
    source_df[DATE_COLUMN],
    errors="coerce"
)

source_df = source_df[
    source_df[CUSTOMER_COLUMN].notna()
].copy()

source_df = source_df[
    source_df[DATE_COLUMN].notna()
].copy()

source_df = source_df.sort_values(
    DATE_COLUMN
).reset_index(drop=True)

print("Source data loaded.")
print(f"Rows: {len(source_df):,}")
print(
    "Customers:",
    source_df[CUSTOMER_COLUMN].nunique()
)
print(
    "Date range:",
    source_df[DATE_COLUMN].min(),
    "to",
    source_df[DATE_COLUMN].max()
)

display(source_df.head())

Source data loaded.
Rows: 99,441
Customers: 96096
Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_orders,total_spend,average_order_value,total_items_purchased,average_review_score,...,last_purchase_date,customer_lifetime_days,purchase_frequency,recency_days,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_score
0,08c5351a6aca1c1589a38f244edeee9d,b7d76e111c89f7ebf14761390f0f7d17,69309,boa vista,RR,1,136.23,136.23,2.0,1.0,...,2016-09-04 21:15:19,0.0,1.0,773.843738,1,136.23,1,5,4,154
1,683c54fc24d40ee9f8a6fc179fd9856c,4854e9b3feff728c13ee5fc7d1547e92,99025,passo fundo,RS,1,75.06,75.06,1.0,1.0,...,2016-09-05 00:15:34,0.0,1.0,773.718565,1,75.06,1,4,2,142
2,622e13439d6b5a0b486c435618b2679e,009b0127b727ab0ba422f6d9604487c7,12244,sao jose dos campos,SP,1,0.00,0.00,0.0,1.0,...,2016-09-13 15:24:19,0.0,1.0,765.087488,1,0.00,1,5,1,151
3,86dc2ffce2dfff336de2f386a786e574,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,1,143.46,143.46,3.0,1.0,...,2016-09-15 12:16:38,0.0,1.0,763.217824,1,143.46,1,2,4,124
4,b106b360fe2ef8849fbbd056f777b4d5,0eb1ee9dba87f5b36b4613a65074337c,2975,sao paulo,SP,1,109.34,109.34,1.0,1.0,...,2016-10-02 22:07:52,0.0,1.0,745.807245,1,109.34,1,5,3,153


In [14]:
duplicate_rows = source_df.duplicated().sum()

missing_customer_values = (
    source_df[CUSTOMER_COLUMN]
    .isna()
    .sum()
)

missing_date_values = (
    source_df[DATE_COLUMN]
    .isna()
    .sum()
)

print("Source validation:")
print(
    "Duplicate rows:",
    duplicate_rows
)
print(
    "Missing customer identifiers:",
    missing_customer_values
)
print(
    "Missing dates:",
    missing_date_values
)

if duplicate_rows > 0:
    print(
        "Warning: duplicate rows detected."
    )

if (
    source_df[CUSTOMER_COLUMN]
    .nunique() < 2
):
    raise RuntimeError(
        "The source does not contain enough customers "
        "for churn modeling."
    )

if (
    source_df[DATE_COLUMN].nunique() < 2
):
    raise RuntimeError(
        "The source does not contain enough date "
        "variation for a historical/future churn design."
    )

Source validation:
Duplicate rows: 0
Missing customer identifiers: 0
Missing dates: 0


In [15]:
MAX_DATE = source_df[DATE_COLUMN].max()
MIN_DATE = source_df[DATE_COLUMN].min()

TOTAL_DAYS = (
    MAX_DATE - MIN_DATE
).days

if TOTAL_DAYS < 180:
    raise RuntimeError(
        "The available dataset spans fewer than 180 days. "
        "A reliable historical churn cutoff and future "
        "observation period cannot be safely created."
    )

FUTURE_HORIZON_DAYS = 90

CUTOFF_DATE = (
    MAX_DATE
    - pd.Timedelta(
        days=FUTURE_HORIZON_DAYS
    )
)

HISTORICAL_DATA = source_df[
    source_df[DATE_COLUMN] < CUTOFF_DATE
].copy()

FUTURE_DATA = source_df[
    source_df[DATE_COLUMN] >= CUTOFF_DATE
].copy()

if HISTORICAL_DATA.empty:
    raise RuntimeError(
        "The historical period contains no data."
    )

if FUTURE_DATA.empty:
    raise RuntimeError(
        "The future observation period contains no data."
    )

print("Temporal design created.")
print(
    "Minimum date:",
    MIN_DATE
)
print(
    "Maximum date:",
    MAX_DATE
)
print(
    "Cutoff date:",
    CUTOFF_DATE
)
print(
    "Historical rows:",
    len(HISTORICAL_DATA)
)
print(
    "Future rows:",
    len(FUTURE_DATA)
)

Temporal design created.
Minimum date: 2016-09-04 21:15:19
Maximum date: 2018-10-17 17:30:18
Cutoff date: 2018-07-19 17:30:18
Historical rows: 89912
Future rows: 9529


In [16]:
historical_customer = (
    HISTORICAL_DATA
    .groupby(CUSTOMER_COLUMN)
    .agg(
        historical_order_count=(
            DATE_COLUMN,
            "count"
        ),
        historical_first_activity=(
            DATE_COLUMN,
            "min"
        ),
        historical_last_activity=(
            DATE_COLUMN,
            "max"
        )
    )
    .reset_index()
)

historical_customer[
    "historical_recency_days"
] = (
    CUTOFF_DATE
    - historical_customer[
        "historical_last_activity"
    ]
).dt.days

historical_customer[
    "historical_lifetime_days"
] = (
    historical_customer[
        "historical_last_activity"
    ]
    - historical_customer[
        "historical_first_activity"
    ]
).dt.days

historical_customer[
    "historical_active_months"
] = (
    historical_customer[
        "historical_lifetime_days"
    ] / 30.44
)

future_activity = (
    FUTURE_DATA
    .groupby(CUSTOMER_COLUMN)
    .agg(
        future_activity_count=(
            DATE_COLUMN,
            "count"
        ),
        future_first_activity=(
            DATE_COLUMN,
            "min"
        )
    )
    .reset_index()
)

customer_model_df = historical_customer.merge(
    future_activity,
    on=CUSTOMER_COLUMN,
    how="left"
)

customer_model_df[
    "future_activity_count"
] = (
    customer_model_df[
        "future_activity_count"
    ]
    .fillna(0)
)

customer_model_df[
    "churned"
] = (
    customer_model_df[
        "future_activity_count"
    ] == 0
).astype(int)

customer_model_df[
    "churn_status"
] = np.where(
    customer_model_df["churned"] == 1,
    "Churned",
    "Active"
)

display(
    customer_model_df.head()
)

,customer_unique_id,historical_order_count,historical_first_activity,historical_last_activity,historical_recency_days,historical_lifetime_days,historical_active_months,future_activity_count,future_first_activity,churned,churn_status
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,70,0,0.0,0.0,NaT,1,Churned
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,73,0,0.0,0.0,NaT,1,Churned
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,495,0,0.0,0.0,NaT,1,Churned
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,279,0,0.0,0.0,NaT,1,Churned
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,246,0,0.0,0.0,NaT,1,Churned


In [17]:
LEAKAGE_KEYWORDS = {
    "future",
    "churn",
    "label",
    "target",
    "outcome",
    "prediction",
    "probability",
    "status"
}

feature_candidates = [
    column
    for column in customer_model_df.columns
    if column not in {
        CUSTOMER_COLUMN,
        "churned",
        "churn_status",
        "future_activity_count",
        "future_first_activity"
    }
]

leakage_columns = []

for column in feature_candidates:

    normalized = normalize_text(
        column
    )

    if any(
        keyword in normalized
        for keyword in LEAKAGE_KEYWORDS
    ):
        leakage_columns.append(column)

if leakage_columns:
    raise RuntimeError(
        "Potential target leakage detected in "
        f"feature columns: {leakage_columns}"
    )

print("Leakage validation passed.")
print(
    "Candidate model features:",
    feature_candidates
)

Leakage validation passed.
Candidate model features: ['historical_order_count', 'historical_first_activity', 'historical_last_activity', 'historical_recency_days', 'historical_lifetime_days', 'historical_active_months']


In [18]:
model_features = [
    "historical_order_count",
    "historical_recency_days",
    "historical_lifetime_days",
    "historical_active_months"
]

missing_model_features = [
    column
    for column in model_features
    if column not in customer_model_df.columns
]

if missing_model_features:
    raise RuntimeError(
        "Required dynamically created historical "
        f"features are missing: {missing_model_features}"
    )

model_df = customer_model_df[
    [
        CUSTOMER_COLUMN,
        *model_features,
        "churned",
        "churn_status"
    ]
].copy()

model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = model_df.drop_duplicates(
    subset=[CUSTOMER_COLUMN]
)

for column in model_features:
    model_df[column] = pd.to_numeric(
        model_df[column],
        errors="coerce"
    )

model_df = model_df.dropna(
    subset=model_features
)

if model_df.empty:
    raise RuntimeError(
        "No valid customer feature rows remain "
        "after validation."
    )

print(
    f"Final modeling rows: {len(model_df):,}"
)

display(
    model_df[model_features + ["churned"]]
    .describe()
)

Final modeling rows: 86,924


,historical_order_count,historical_recency_days,historical_lifetime_days,historical_active_months,churned
count,86924.000000,86924.00000,86924.000000,86924.000000,86924.000000
mean,1.034375,221.12129,2.461748,0.080872,0.997239
std,0.210656,142.94833,23.437215,0.769948,0.052473
min,1.000000,0.00000,0.000000,0.000000,0.000000
25%,1.000000,104.00000,0.000000,0.000000,1.000000
50%,1.000000,199.00000,0.000000,0.000000,1.000000
75%,1.000000,324.00000,0.000000,0.000000,1.000000
max,13.000000,682.00000,633.000000,20.795007,1.000000


In [19]:
class_distribution = (
    model_df["churn_status"]
    .value_counts()
    .rename_axis("class")
    .reset_index(
        name="customer_count"
    )
)

class_distribution[
    "percentage"
] = (
    class_distribution[
        "customer_count"
    ]
    / len(model_df)
    * 100
)

display(class_distribution)

if model_df["churned"].nunique() < 2:
    raise RuntimeError(
        "The churn target contains only one class. "
        "A classification model cannot be trained."
    )

if (
    model_df["churned"].value_counts()
    .min()
    < 5
):
    raise RuntimeError(
        "One churn class contains fewer than five "
        "customers. The dataset is too small for "
        "reliable stratified model evaluation."
    )

,class,customer_count,percentage
0,Churned,86684,99.723897
1,Active,240,0.276103


In [20]:
X = model_df[
    model_features
].copy()

y = model_df[
    "churned"
].copy()

customer_ids = model_df[
    CUSTOMER_COLUMN
].copy()

X_train, X_temp, y_train, y_temp, ids_train, ids_temp = (
    train_test_split(
        X,
        y,
        customer_ids,
        test_size=0.30,
        stratify=y,
        random_state=42
    )
)

X_validation, X_test, y_validation, y_test, ids_validation, ids_test = (
    train_test_split(
        X_temp,
        y_temp,
        ids_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )
)

print("Data split completed.")
print(
    "Training rows:",
    len(X_train)
)
print(
    "Validation rows:",
    len(X_validation)
)
print(
    "Test rows:",
    len(X_test)
)

print("\nTraining class distribution:")
display(
    y_train.value_counts(
        normalize=True
    )
)

print("\nValidation class distribution:")
display(
    y_validation.value_counts(
        normalize=True
    )
)

print("\nTest class distribution:")
display(
    y_test.value_counts(
        normalize=True
    )
)

Data split completed.
Training rows: 60846
Validation rows: 13039
Test rows: 13039

Training class distribution:


churned
1    0.997239
0    0.002761
Name: proportion, dtype: float64


Validation class distribution:


churned
1    0.997239
0    0.002761
Name: proportion, dtype: float64


Test class distribution:


churned
1    0.997239
0    0.002761
Name: proportion, dtype: float64

In [21]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            model_features
        )
    ],
    remainder="drop"
)

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=400,
                class_weight="balanced",
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

print(
    "Leakage-safe models created:"
)

for model_name in models:
    print(
        f"- {model_name}"
    )

Leakage-safe models created:
- Logistic Regression
- Random Forest


In [22]:
def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.5
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        )
    }

    if len(
        np.unique(y_true)
    ) == 2:
        metrics["roc_auc"] = roc_auc_score(
            y_true,
            probabilities
        )
    else:
        metrics["roc_auc"] = np.nan

    metrics[
        "predicted_churned"
    ] = int(
        predictions.sum()
    )

    metrics[
        "predicted_active"
    ] = int(
        len(predictions)
        - predictions.sum()
    )

    metrics[
        "predicts_only_one_class"
    ] = (
        len(
            np.unique(predictions)
        ) == 1
    )

    return metrics


validation_results = []
trained_models = {}

for model_name, model in models.items():

    fitted_model = clone(model)

    fitted_model.fit(
        X_train,
        y_train
    )

    validation_probabilities = (
        fitted_model.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_metrics = (
        calculate_metrics(
            y_validation,
            validation_probabilities,
            threshold=0.5
        )
    )

    validation_metrics[
        "model"
    ] = model_name

    validation_results.append(
        validation_metrics
    )

    trained_models[
        model_name
    ] = fitted_model

validation_results_df = pd.DataFrame(
    validation_results
)

display(
    validation_results_df
)

,accuracy,precision,recall,f1,roc_auc,predicted_churned,predicted_active,predicts_only_one_class,model
0,0.595751,0.997811,0.595939,0.746208,0.622495,7766,5273,False,Logistic Regression
1,0.864637,0.997521,0.866415,0.927357,0.658293,11294,1745,False,Random Forest


In [23]:
def find_best_threshold(
    y_true,
    probabilities
):
    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probabilities
        )
    )

    if len(thresholds) == 0:
        return 0.5

    f1_scores = (
        2
        * precision[:-1]
        * recall[:-1]
        / (
            precision[:-1]
            + recall[:-1]
            + 1e-12
        )
    )

    best_index = int(
        np.argmax(f1_scores)
    )

    return float(
        thresholds[best_index]
    )


threshold_results = []

for model_name, model in trained_models.items():

    validation_probabilities = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    best_threshold = (
        find_best_threshold(
            y_validation,
            validation_probabilities
        )
    )

    threshold_predictions = (
        validation_probabilities
        >= best_threshold
    ).astype(int)

    threshold_metrics = (
        calculate_metrics(
            y_validation,
            validation_probabilities,
            threshold=best_threshold
        )
    )

    threshold_metrics[
        "model"
    ] = model_name

    threshold_metrics[
        "threshold"
    ] = best_threshold

    threshold_results.append(
        threshold_metrics
    )

threshold_results_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_results_df
)

,accuracy,precision,recall,f1,roc_auc,predicted_churned,predicted_active,predicts_only_one_class,model,threshold
0,0.997239,0.997239,1.0,0.998618,0.622495,13039,0,True,Logistic Regression,0.002151
1,0.997239,0.997239,1.0,0.998618,0.658293,13039,0,True,Random Forest,0.106184


In [26]:
# Cell 21 — Robust model selection and threshold validation

model_selection_results = []

for model_name, model in trained_models.items():

    # Generate validation probabilities
    validation_probabilities = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    # Check probability variation
    probability_min = (
        validation_probabilities.min()
    )

    probability_max = (
        validation_probabilities.max()
    )

    probability_mean = (
        validation_probabilities.mean()
    )

    probability_std = (
        validation_probabilities.std()
    )

    probability_unique = (
        len(
            np.unique(
                validation_probabilities
            )
        )
    )

    # Test a wide range of thresholds.
    # This prevents the model from being judged
    # only at the default 0.50 threshold.
    candidate_thresholds = np.linspace(
        0.05,
        0.95,
        181
    )

    best_model_threshold = None
    best_model_f1 = -1
    best_model_precision = 0
    best_model_recall = 0
    best_model_accuracy = 0
    best_predicted_classes = None
    best_predicted_churned = 0
    best_predicted_active = 0

    for threshold in candidate_thresholds:

        threshold_predictions = (
            validation_probabilities
            >= threshold
        ).astype(int)

        predicted_classes = (
            np.unique(
                threshold_predictions
            )
        )

        predicted_churned = int(
            threshold_predictions.sum()
        )

        predicted_active = int(
            len(threshold_predictions)
            - predicted_churned
        )

        current_f1 = f1_score(
            y_validation,
            threshold_predictions,
            zero_division=0
        )

        current_precision = (
            precision_score(
                y_validation,
                threshold_predictions,
                zero_division=0
            )
        )

        current_recall = (
            recall_score(
                y_validation,
                threshold_predictions,
                zero_division=0
            )
        )

        current_accuracy = (
            accuracy_score(
                y_validation,
                threshold_predictions
            )
        )

        # Only accept thresholds that predict
        # both classes.
        if len(
            predicted_classes
        ) < 2:

            continue

        # Select the threshold with the best F1.
        # If tied, prefer better recall.
        if (
            current_f1 > best_model_f1
            or (
                current_f1 == best_model_f1
                and current_recall
                > best_model_recall
            )
        ):

            best_model_threshold = (
                threshold
            )

            best_model_f1 = (
                current_f1
            )

            best_model_precision = (
                current_precision
            )

            best_model_recall = (
                current_recall
            )

            best_model_accuracy = (
                current_accuracy
            )

            best_predicted_classes = (
                predicted_classes
            )

            best_predicted_churned = (
                predicted_churned
            )

            best_predicted_active = (
                predicted_active
            )

    # Calculate ROC-AUC separately.
    if (
        len(
            np.unique(
                y_validation
            )
        )
        == 2
    ):

        validation_roc_auc = (
            roc_auc_score(
                y_validation,
                validation_probabilities
            )
        )

    else:

        validation_roc_auc = np.nan


    # If no threshold can produce both classes,
    # mark the model as degenerate.
    if (
        best_model_threshold
        is None
    ):

        model_selection_results.append(
            {
                "model": model_name,
                "probability_min": (
                    probability_min
                ),
                "probability_max": (
                    probability_max
                ),
                "probability_mean": (
                    probability_mean
                ),
                "probability_std": (
                    probability_std
                ),
                "unique_probability_values": (
                    probability_unique
                ),
                "validation_accuracy": np.nan,
                "validation_precision": np.nan,
                "validation_recall": np.nan,
                "validation_f1": np.nan,
                "validation_roc_auc": (
                    validation_roc_auc
                ),
                "threshold": np.nan,
                "predicted_active": 0,
                "predicted_churned": 0,
                "predicted_classes": [],
                "predicts_both_classes": False
            }
        )

    else:

        model_selection_results.append(
            {
                "model": model_name,
                "probability_min": (
                    probability_min
                ),
                "probability_max": (
                    probability_max
                ),
                "probability_mean": (
                    probability_mean
                ),
                "probability_std": (
                    probability_std
                ),
                "unique_probability_values": (
                    probability_unique
                ),
                "validation_accuracy": (
                    best_model_accuracy
                ),
                "validation_precision": (
                    best_model_precision
                ),
                "validation_recall": (
                    best_model_recall
                ),
                "validation_f1": (
                    best_model_f1
                ),
                "validation_roc_auc": (
                    validation_roc_auc
                ),
                "threshold": (
                    best_model_threshold
                ),
                "predicted_active": (
                    best_predicted_active
                ),
                "predicted_churned": (
                    best_predicted_churned
                ),
                "predicted_classes": (
                    best_predicted_classes.tolist()
                ),
                "predicts_both_classes": True
            }
        )


model_selection_df = pd.DataFrame(
    model_selection_results
)


print(
    "Model selection diagnostics:"
)

display(
    model_selection_df
)


# Keep only models that can produce
# both Active and Churned predictions.
valid_models = (
    model_selection_df[
        model_selection_df[
            "predicts_both_classes"
        ]
        == True
    ]
    .copy()
)


if valid_models.empty:

    raise RuntimeError(
        "MODEL FAILURE: None of the trained "
        "models can produce both Active and "
        "Churned predictions at any tested "
        "validation threshold."
    )


# Select the model using validation F1,
# with ROC-AUC as a secondary criterion.
valid_models = (
    valid_models
    .sort_values(
        [
            "validation_f1",
            "validation_roc_auc",
            "validation_recall"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


best_model_name = (
    valid_models.iloc[0]["model"]
)


best_threshold = float(
    valid_models.iloc[0]["threshold"]
)


best_model = trained_models[
    best_model_name
]


print(
    "Selected model:",
    best_model_name
)

print(
    "Selected threshold:",
    best_threshold
)

print(
    "Validation F1:",
    valid_models.iloc[0][
        "validation_f1"
    ]
)

print(
    "Validation ROC-AUC:",
    valid_models.iloc[0][
        "validation_roc_auc"
    ]
)

print(
    "Validation predicted Active:",
    int(
        valid_models.iloc[0][
            "predicted_active"
        ]
    )
)

print(
    "Validation predicted Churned:",
    int(
        valid_models.iloc[0][
            "predicted_churned"
        ]
    )
)

Model selection diagnostics:


,model,probability_min,probability_max,probability_mean,probability_std,unique_probability_values,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,threshold,predicted_active,predicted_churned,predicted_classes,predicts_both_classes
0,Logistic Regression,0.002151,0.754355,0.528047,0.091052,949,0.996702,0.997314,0.999385,0.998348,0.622495,0.05,9,13030,"[0, 1]",True
1,Random Forest,0.106184,1.000000,0.854295,0.239797,436,0.995782,0.997235,0.998539,0.997886,0.658293,0.11,19,13020,"[0, 1]",True


Selected model: Logistic Regression
Selected threshold: 0.05
Validation F1: 0.998348250297699
Validation ROC-AUC: 0.6224952361420869
Validation predicted Active: 9
Validation predicted Churned: 13030


In [27]:
test_probabilities = (
    best_model.predict_proba(
        X_test
    )[:, 1]
)

test_predictions = (
    test_probabilities
    >= best_threshold
).astype(int)

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    threshold=best_threshold
)

test_metrics_df = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "threshold": best_threshold,
            **test_metrics
        }
    ]
)

display(
    test_metrics_df
)

print("\nConfusion matrix:")

test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions
)

confusion_matrix_df = pd.DataFrame(
    test_confusion_matrix,
    index=[
        "Actual Active",
        "Actual Churned"
    ],
    columns=[
        "Predicted Active",
        "Predicted Churned"
    ]
)

display(
    confusion_matrix_df
)

print("\nClassification report:")

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "Active",
            "Churned"
        ],
        zero_division=0
    )
)

,model,threshold,accuracy,precision,recall,f1,roc_auc,predicted_churned,predicted_active,predicts_only_one_class
0,Logistic Regression,0.05,0.996932,0.997315,0.999615,0.998464,0.63361,13033,6,False



Confusion matrix:


,Predicted Active,Predicted Churned
Actual Active,1,35
Actual Churned,5,12998



Classification report:
              precision    recall  f1-score   support

      Active       0.17      0.03      0.05        36
     Churned       1.00      1.00      1.00     13003

    accuracy                           1.00     13039
   macro avg       0.58      0.51      0.52     13039
weighted avg       1.00      1.00      1.00     13039



In [28]:
if test_metrics[
    "predicts_only_one_class"
]:
    raise RuntimeError(
        "FINAL MODEL FAILURE: The selected model "
        "predicts only one class on the test set."
    )

if (
    test_metrics["predicted_churned"] == 0
    or test_metrics["predicted_active"] == 0
):
    raise RuntimeError(
        "FINAL MODEL FAILURE: The final prediction "
        "distribution contains only one class."
    )

print(
    "Final prediction-distribution validation passed."
)

print(
    "Predicted Active:",
    test_metrics["predicted_active"]
)

print(
    "Predicted Churned:",
    test_metrics["predicted_churned"]
)

print(
    "ROC-AUC:",
    test_metrics["roc_auc"]
)

Final prediction-distribution validation passed.
Predicted Active: 6
Predicted Churned: 13033
ROC-AUC: 0.633610192519675


In [29]:
all_probabilities = (
    best_model.predict_proba(
        X
    )[:, 1]
)

all_predictions = (
    all_probabilities
    >= best_threshold
).astype(int)

prediction_output = model_df[
    [
        CUSTOMER_COLUMN
    ]
].copy()

prediction_output[
    "churn_probability"
] = all_probabilities

prediction_output[
    "churn_prediction"
] = all_predictions

prediction_output[
    "churn_status"
] = np.where(
    all_predictions == 1,
    "Churned",
    "Active"
)

prediction_output[
    "risk_level"
] = pd.cut(
    prediction_output[
        "churn_probability"
    ],
    bins=[
        -np.inf,
        0.33,
        0.66,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

prediction_output[
    "prediction_cutoff_date"
] = CUTOFF_DATE

prediction_output[
    "future_horizon_days"
] = FUTURE_HORIZON_DAYS

prediction_output[
    "model_name"
] = best_model_name

prediction_output[
    "model_threshold"
] = best_threshold

prediction_output = (
    prediction_output
    .sort_values(
        "churn_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    prediction_output.head(20)
)

,customer_unique_id,churn_probability,churn_prediction,churn_status,risk_level,prediction_cutoff_date,future_horizon_days,model_name,model_threshold
0,b7d76e111c89f7ebf14761390f0f7d17,0.766218,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
1,4854e9b3feff728c13ee5fc7d1547e92,0.766218,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
2,009b0127b727ab0ba422f6d9604487c7,0.762985,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
3,830d5b7aaa3b6f1e9ad63703bec97d23,0.762173,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
4,2f64e403852e6893ae37485d5fcacdaf,0.754771,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
5,0eb1ee9dba87f5b36b4613a65074337c,0.754771,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
6,8d3a54507421dbd2ce0a1d58046826e0,0.754355,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
7,823c47d4abda1f8ce7568145f76c2b85,0.754355,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
8,b8b8726af116a5cfb35b0315ecef9172,0.754355,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05
9,8de8484141a728d73763be267b73cda2,0.754355,1,Churned,High,2018-07-19 17:30:18,90,Logistic Regression,0.05


In [30]:
final_prediction_distribution = (
    prediction_output[
        "churn_status"
    ]
    .value_counts()
    .rename_axis("churn_status")
    .reset_index(
        name="customer_count"
    )
)

final_prediction_distribution[
    "percentage"
] = (
    final_prediction_distribution[
        "customer_count"
    ]
    / len(prediction_output)
    * 100
)

display(
    final_prediction_distribution
)

if (
    prediction_output[
        "churn_status"
    ].nunique()
    < 2
):
    raise RuntimeError(
        "Final customer predictions contain only "
        "one churn status."
    )

,churn_status,customer_count,percentage
0,Churned,86861,99.927523
1,Active,63,0.072477


In [31]:
prediction_file = (
    PREDICTIONS_PATH
    / "customer_churn_predictions.csv"
)

prediction_output.to_csv(
    prediction_file,
    index=False,
    encoding="utf-8-sig"
)

if not prediction_file.exists():
    raise RuntimeError(
        "Prediction CSV was not created."
    )

print(
    "Prediction CSV saved successfully:"
)

print(
    prediction_file
)

print(
    "Rows saved:",
    len(prediction_output)
)

Prediction CSV saved successfully:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\customer_churn_predictions.csv
Rows saved: 86924


In [32]:
model_file = (
    MODELS_PATH
    / "customer_churn_model.joblib"
)

model_artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "threshold": best_threshold,
    "source_schema": SOURCE_SCHEMA,
    "source_table": SOURCE_TABLE,
    "customer_column": CUSTOMER_COLUMN,
    "date_column": DATE_COLUMN,
    "model_features": model_features,
    "cutoff_date": str(CUTOFF_DATE),
    "future_horizon_days": FUTURE_HORIZON_DAYS,
    "created_at": datetime.now().isoformat()
}

joblib.dump(
    model_artifact,
    model_file
)

if not model_file.exists():
    raise RuntimeError(
        "Model file was not created."
    )

print(
    "Trained model saved successfully:"
)

print(
    model_file
)

Trained model saved successfully:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\models\customer_churn_model.joblib


In [33]:
model_comparison_output = (
    threshold_results_df
    .copy()
)

model_comparison_file = (
    PREDICTIONS_PATH
    / "customer_churn_model_comparison.csv"
)

model_comparison_output.to_csv(
    model_comparison_file,
    index=False,
    encoding="utf-8-sig"
)

test_metrics_file = (
    PREDICTIONS_PATH
    / "customer_churn_test_metrics.csv"
)

test_metrics_df.to_csv(
    test_metrics_file,
    index=False,
    encoding="utf-8-sig"
)

prediction_distribution_file = (
    PREDICTIONS_PATH
    / "customer_churn_prediction_distribution.csv"
)

final_prediction_distribution.to_csv(
    prediction_distribution_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Additional Power BI-ready validation outputs saved:"
)

print(model_comparison_file)
print(test_metrics_file)
print(prediction_distribution_file)

Additional Power BI-ready validation outputs saved:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\customer_churn_model_comparison.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\customer_churn_test_metrics.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\customer_churn_prediction_distribution.csv


In [34]:
def make_safe_table_name(value):
    value = normalize_text(value)

    if not value:
        raise ValueError(
            "Generated PostgreSQL table name is empty."
        )

    return value


POSTGRES_OUTPUT_TABLE = make_safe_table_name(
    "customer_churn_predictions"
)

POSTGRES_METRICS_TABLE = make_safe_table_name(
    "customer_churn_model_metrics"
)

POSTGRES_DISTRIBUTION_TABLE = make_safe_table_name(
    "customer_churn_prediction_distribution"
)

print(
    "PostgreSQL output tables:"
)

print(
    POSTGRES_OUTPUT_TABLE
)

print(
    POSTGRES_METRICS_TABLE
)

print(
    POSTGRES_DISTRIBUTION_TABLE
)

PostgreSQL output tables:
customer_churn_predictions
customer_churn_model_metrics
customer_churn_prediction_distribution


In [35]:
OUTPUT_SCHEMA = "analytics"

with connection.cursor() as cursor:

    cursor.execute(
        sql.SQL(
            """
            CREATE SCHEMA IF NOT EXISTS {}
            """
        ).format(
            quote_identifier(OUTPUT_SCHEMA)
        )
    )

connection.commit()

print(
    f"Output schema available: {OUTPUT_SCHEMA}"
)

Output schema available: analytics


In [38]:
# Cell 31 — Write final churn predictions and metrics to PostgreSQL

import pandas as pd

from getpass import getpass
from urllib.parse import quote_plus

from sqlalchemy import (
    create_engine,
    text
)


# ---------------------------------------------------------
# 1. PostgreSQL connection settings
# ---------------------------------------------------------

POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_DATABASE = "ecommerce_ai_db"
POSTGRES_USER = "postgres"


# ---------------------------------------------------------
# 2. Securely request the PostgreSQL password
# ---------------------------------------------------------

POSTGRES_PASSWORD = getpass(
    "Enter your PostgreSQL password: "
)


# URL-encode the password so special characters
# do not break the connection string.
encoded_password = quote_plus(
    POSTGRES_PASSWORD
)


# ---------------------------------------------------------
# 3. Create SQLAlchemy PostgreSQL engine
# ---------------------------------------------------------

POSTGRES_CONNECTION_STRING = (
    "postgresql+psycopg2://"
    f"{POSTGRES_USER}:"
    f"{encoded_password}@"
    f"{POSTGRES_HOST}:"
    f"{POSTGRES_PORT}/"
    f"{POSTGRES_DATABASE}"
)


sqlalchemy_engine = create_engine(
    POSTGRES_CONNECTION_STRING
)


# ---------------------------------------------------------
# 4. Validate the SQLAlchemy connection
# ---------------------------------------------------------

with sqlalchemy_engine.connect() as test_connection:

    test_connection.execute(
        text(
            "SELECT 1"
        )
    )


print(
    "SQLAlchemy PostgreSQL connection "
    "validated successfully."
)


# ---------------------------------------------------------
# 5. Prepare prediction output
# ---------------------------------------------------------

prediction_db_output = (
    prediction_output.copy()
)


if (
    "prediction_cutoff_date"
    in prediction_db_output.columns
):

    prediction_db_output[
        "prediction_cutoff_date"
    ] = pd.to_datetime(
        prediction_db_output[
            "prediction_cutoff_date"
        ]
    )


# ---------------------------------------------------------
# 6. Write churn predictions to PostgreSQL
# ---------------------------------------------------------

prediction_db_output.to_sql(
    name=POSTGRES_OUTPUT_TABLE,
    con=sqlalchemy_engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi"
)


# ---------------------------------------------------------
# 7. Prepare model metrics output
# ---------------------------------------------------------

test_metrics_db_output = (
    test_metrics_df.copy()
)


# ---------------------------------------------------------
# 8. Write model metrics to PostgreSQL
# ---------------------------------------------------------

test_metrics_db_output.to_sql(
    name=POSTGRES_METRICS_TABLE,
    con=sqlalchemy_engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi"
)


print(
    "Final churn predictions written to PostgreSQL:"
)

print(
    f"{OUTPUT_SCHEMA}."
    f"{POSTGRES_OUTPUT_TABLE}"
)


print(
    "Model evaluation metrics written to PostgreSQL:"
)

print(
    f"{OUTPUT_SCHEMA}."
    f"{POSTGRES_METRICS_TABLE}"
)

Enter your PostgreSQL password:  ········


SQLAlchemy PostgreSQL connection validated successfully.
Final churn predictions written to PostgreSQL:
analytics.customer_churn_predictions
Model evaluation metrics written to PostgreSQL:
analytics.customer_churn_model_metrics


In [41]:
# Cell 32 — Verify PostgreSQL output tables safely

# ---------------------------------------------------------
# 1. Recover the raw PostgreSQL connection
# ---------------------------------------------------------

if connection.closed:

    raise RuntimeError(
        "The PostgreSQL connection is closed. "
        "Reconnect before running Cell 32."
    )


# The previous failed SQL statement may have
# aborted the current transaction.
connection.rollback()


# ---------------------------------------------------------
# 2. Prepare verification
# ---------------------------------------------------------

verification_results = []


verification_tables = [
    POSTGRES_OUTPUT_TABLE,
    POSTGRES_METRICS_TABLE
]


# ---------------------------------------------------------
# 3. Verify each actual output table
# ---------------------------------------------------------

with connection.cursor() as cursor:

    for table_name in verification_tables:

        cursor.execute(
            """
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_schema = %s
                  AND table_name = %s
            )
            """,
            (
                OUTPUT_SCHEMA,
                table_name
            )
        )


        table_exists = (
            cursor.fetchone()[0]
        )


        if not table_exists:

            verification_results.append(
                {
                    "schema": OUTPUT_SCHEMA,
                    "table": table_name,
                    "exists": False,
                    "row_count": None,
                    "status": "MISSING"
                }
            )

            continue


        # Use dynamically quoted identifiers
        # only after confirming the table exists.
        cursor.execute(
            sql.SQL(
                """
                SELECT COUNT(*)
                FROM {}.{}
                """
            ).format(
                quote_identifier(
                    OUTPUT_SCHEMA
                ),
                quote_identifier(
                    table_name
                )
            )
        )


        row_count = (
            cursor.fetchone()[0]
        )


        verification_results.append(
            {
                "schema": OUTPUT_SCHEMA,
                "table": table_name,
                "exists": True,
                "row_count": row_count,
                "status": "VERIFIED"
            }
        )


# ---------------------------------------------------------
# 4. Commit the successful verification transaction
# ---------------------------------------------------------

connection.commit()


# ---------------------------------------------------------
# 5. Display verification results
# ---------------------------------------------------------

verification_results_df = pd.DataFrame(
    verification_results
)


display(
    verification_results_df
)


# ---------------------------------------------------------
# 6. Stop if an expected output is missing
# ---------------------------------------------------------

missing_tables = (
    verification_results_df[
        verification_results_df[
            "exists"
        ]
        == False
    ]
)


if not missing_tables.empty:

    raise RuntimeError(
        "One or more expected PostgreSQL "
        "output tables are missing."
    )


print(
    "PostgreSQL output-table verification "
    "completed successfully."
)

,schema,table,exists,row_count,status
0,analytics,customer_churn_predictions,True,86924,VERIFIED
1,analytics,customer_churn_model_metrics,True,1,VERIFIED


PostgreSQL output-table verification completed successfully.


In [42]:
print("=" * 70)
print("NOTEBOOK 11 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nSOURCE")
print(
    f"{SOURCE_SCHEMA}.{SOURCE_TABLE}"
)

print(
    "Customer identifier:",
    CUSTOMER_COLUMN
)

print(
    "Date column:",
    DATE_COLUMN
)

print("\nTEMPORAL DESIGN")
print(
    "Cutoff date:",
    CUTOFF_DATE
)

print(
    "Future horizon:",
    FUTURE_HORIZON_DAYS,
    "days"
)

print("\nMODEL")
print(
    "Selected model:",
    best_model_name
)

print(
    "Threshold:",
    best_threshold
)

print("\nFINAL TEST METRICS")
display(
    test_metrics_df
)

print("\nFINAL PREDICTION DISTRIBUTION")
display(
    final_prediction_distribution
)

print("\nFILES")
print(
    prediction_file
)

print(
    model_file
)

print("\nPOSTGRESQL")
print(
    f"{OUTPUT_SCHEMA}.{POSTGRES_OUTPUT_TABLE}"
)

print(
    f"{OUTPUT_SCHEMA}.{POSTGRES_METRICS_TABLE}"
)

print(
    f"{OUTPUT_SCHEMA}.{POSTGRES_DISTRIBUTION_TABLE}"
)

NOTEBOOK 11 COMPLETED SUCCESSFULLY

SOURCE
feature_engineered.customer_features
Customer identifier: customer_unique_id
Date column: first_purchase_date

TEMPORAL DESIGN
Cutoff date: 2018-07-19 17:30:18
Future horizon: 90 days

MODEL
Selected model: Logistic Regression
Threshold: 0.05

FINAL TEST METRICS


,model,threshold,accuracy,precision,recall,f1,roc_auc,predicted_churned,predicted_active,predicts_only_one_class
0,Logistic Regression,0.05,0.996932,0.997315,0.999615,0.998464,0.63361,13033,6,False



FINAL PREDICTION DISTRIBUTION


,churn_status,customer_count,percentage
0,Churned,86861,99.927523
1,Active,63,0.072477



FILES
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\customer_churn_predictions.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\models\customer_churn_model.joblib

POSTGRESQL
analytics.customer_churn_predictions
analytics.customer_churn_model_metrics
analytics.customer_churn_prediction_distribution
